# Depth To Normal Translator

## Import and Setup

In [1]:
import genesis as gs
import numpy as np
from pathlib import Path
import cv2
from scipy.spatial.transform import Rotation as R
from trimesh.transformations import quaternion_from_euler, euler_from_quaternion
import random

from utils import *

[I 04/01/25 17:07:51.674 143903] [shell.py:_shell_pop_print@23] Graphical python shell detected, using wrapped sys.stdout


In [2]:
# Genesis Setup
gs.init(backend=gs.cuda, precision='32', theme='dark', eps=1e-12)
scene = gs.Scene(
    viewer_options=gs.options.ViewerOptions(camera_pos=(1, 1, 1), camera_lookat=(0, 0, 0.15), camera_fov=30, max_FPS=600),
    sim_options=gs.options.SimOptions(dt=0.001),
    show_viewer=True,
    show_FPS=False
)

plane = scene.add_entity(gs.morphs.Plane())
cylinder = scene.add_entity(
    gs.morphs.Cylinder(
        pos=[0.0, 0.0, 0.05],
        height=0.1, 
        radius=0.03,
        collision=False,
        fixed=True
    )
)
spot_gripper = scene.add_entity(
    gs.morphs.URDF(
        file='urdf/spot_arm/urdf/open_gripper.urdf',
        euler=(90, 0, 0),
        pos=(-1, 0.0, 0.10),
        scale=1.0,
        merge_fixed_links=True,
        fixed=True
    )
)

camera = scene.add_entity(
    gs.morphs.Cylinder(
        pos=[0.0, 0.0, 1.0],
        height=0.1, 
        radius=0.02,
        collision=False,
        fixed=True, 
    )
)

cam = scene.add_camera(
    pos=[0.0, 0.0, 1.0],
    lookat = [0.0, 0.0, 0.15],
    res    = (1280, 960),
    fov    = 30,
    GUI    = False,
)

# build 
scene.build()

[Genesis] [17:07:55] [INFO] ╭───────────────────────────────────────────────╮
[Genesis] [17:07:55] [INFO] │┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈ Genesis ┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈│
[Genesis] [17:07:55] [INFO] ╰───────────────────────────────────────────────╯
[Genesis] [17:07:55] [INFO] Running on [NVIDIA GeForce RTX 3060 Laptop GPU] with backend gs.cuda. Device memory: 5.69 GB.
[Genesis] [17:07:55] [INFO] 🚀 Genesis initialized. 🔖 version: 0.2.1, 🌱 seed: None, 📏 precision: '32', 🐛 debug: False, 🎨 theme: 'dark'.
[Genesis] [17:07:55] [INFO] Scene <28300f7> created.
[Genesis] [17:07:55] [INFO] Adding <gs.RigidEntity>. idx: 0, uid: <2fd871c>, morph: <gs.morphs.Plane>, material: <gs.materials.Rigid>.
[Genesis] [17:07:55] [INFO] Adding <gs.RigidEntity>. idx: 1, uid: <54eca78>, morph: <gs.morphs.Cylinder>, material: <gs.materials.Rigid>.
[Genesis] [17:07:55] [INFO] Adding <gs.RigidEntity>. idx: 2, uid: <e97cd8a>, morph: <gs.morphs.URDF(file='/home/de4lerr/Documents/LABROM/labrom_venv/lib/python3.12/site-packages/genes

/home/de4lerr/Documents/LABROM/labrom_venv/lib/python3.12/site-packages/genesis/ext/pyrender/viewer.py


In [3]:
# Define DOFs
hand_dof = np.arange(2)
finger_dof = np.array([1]) 

# Set PD control parameters
spot_gripper.set_dofs_kp(
    np.array([100]*2)
    )
spot_gripper.set_dofs_kv(
    np.array([1]*2)
    )
spot_gripper.set_dofs_force_range(
    np.array([-100]*2),
    np.array([100]*2)
    )

### Function

In [4]:
# D2NT Functions
def compute_d2nt_normal(depth_path, normal_path, VERSION="d2nt_v3"):
    
    # Fake camera intrinsics (adjust based on Genesis camera if known)
    cam_fx, cam_fy, u0, v0 = 1280 / (2 * np.tan(np.deg2rad(30) / 2)), 960 / (2 * np.tan(np.deg2rad(30) / 2)), 640, 480

    # get ground truth normal [-1,1]
    normal_gt = get_normal_gt(normal_path)
    normal_gt = vector_normalization(normal_gt)
    h, w, _ = normal_gt.shape

    # get depth
    depth, mask = get_depth(depth_path, h, w) 
    u_map = np.ones((h, 1)) * np.arange(1, w + 1) - u0
    v_map = np.arange(1, h + 1).reshape(h, 1) * np.ones((1, w)) - v0

    # get depth gradients
    if VERSION == 'd2nt_basic':
        Gu, Gv = get_filter(depth)
    else:
        Gu, Gv = get_DAG_filter(depth)

    # Depth to Normal Translation
    est_nx = Gu * cam_fx
    est_ny = Gv * cam_fy
    est_nz = -(depth + v_map * Gv + u_map * Gu)
    est_normal = cv2.merge((est_nx, est_ny, est_nz))

    # vector normalization
    est_normal = vector_normalization(est_normal)

    # MRF-based Normal Refinement
    if VERSION == 'd2nt_v3':
        est_normal = MRF_optim(depth, est_normal)

    return est_normal

def d2nt(depth, VERSION="d2nt_v3"):
    
    # Fake camera intrinsics (adjust based on Genesis camera if known)
    cam_fx, cam_fy, u0, v0 = 1280 / (2 * np.tan(np.deg2rad(30) / 2)), 960 / (2 * np.tan(np.deg2rad(30) / 2)), 640, 480
    h, w = depth.shape

    # get depth
    u_map = np.ones((h, 1)) * np.arange(1, w + 1) - u0
    v_map = np.arange(1, h + 1).reshape(h, 1) * np.ones((1, w)) - v0

    # get depth gradients
    if VERSION == 'd2nt_basic':
        Gu, Gv = get_filter(depth)
    else:
        Gu, Gv = get_DAG_filter(depth)

    # Depth to Normal Translation
    est_nx = Gu * cam_fx
    est_ny = Gv * cam_fy
    est_nz = -(depth + v_map * Gv + u_map * Gu)
    est_normal = cv2.merge((est_nx, est_ny, est_nz))

    # vector normalization
    est_normal = vector_normalization(est_normal)

    # MRF-based Normal Refinement
    if VERSION == 'd2nt_v3':
        est_normal = MRF_optim(depth, est_normal)

    return est_normal

In [5]:
# Convert to world coordinates
def pixel_to_world(pixel_x, pixel_y, depth, cam_pos, cam_lookat, fov, img_width, img_height, world_up):

    cam_pos = np.array(cam_pos)
    cam_lookat = np.array(cam_lookat)
    
    # Calculate camera direction vector
    cam_dir = cam_lookat - cam_pos
    cam_dir = cam_dir / np.linalg.norm(cam_dir)
    
    # Calculate camera up vector (assuming world up is (0,0,1))
    world_up = np.array(world_up)
    cam_right = np.cross(cam_dir, world_up)
    cam_right = cam_right / np.linalg.norm(cam_right)
    cam_up = np.cross(cam_right, cam_dir)
    
    # Convert FOV to radians
    fov_rad = np.deg2rad(fov)
    
    # Calculate pixel coordinates in normalized device coordinates (-1 to 1)
    ndc_x = (2.0 * pixel_x / (img_width - 1)) - 1.0
    ndc_y = 1.0 - (2.0 * pixel_y / (img_height - 1))  # Flip y-axis
    
    # Calculate direction vector in camera space
    aspect_ratio = img_width / img_height
    tan_half_fov = np.tan(fov_rad / 2.0)
    
    cam_space_x = ndc_x * tan_half_fov * aspect_ratio
    cam_space_y = ndc_y * tan_half_fov
    cam_space_z = -1.0  # Negative z is forward in camera space
    
    # Create direction vector in camera space
    cam_space_dir = np.array([cam_space_x, cam_space_y, cam_space_z])
    cam_space_dir = cam_space_dir / np.linalg.norm(cam_space_dir)
    
    # Convert to world space direction
    world_dir = (cam_right * cam_space_dir[0] + 
                cam_up * cam_space_dir[1] + 
                -cam_dir * cam_space_dir[2])
    world_dir = world_dir / np.linalg.norm(world_dir)
    
    # Calculate world position
    world_pos = cam_pos + world_dir * depth
    
    return world_pos

In [6]:
# Rotation

def normalize(v):
    return v / np.linalg.norm(v)

def rotate_vector(normal, cam_pos, cam_lookat, up):
    # Vetor direção da câmera (câmera olha de cam_pos para cam_lookat)
    z_cam = normalize(np.array(cam_pos) - np.array(cam_lookat))
    
    # Vetor "para cima" definido pelo usuário
    up = np.array(up)
    
    # Vetor X da câmera, perpendicular ao Z (produto vetorial de up e z_cam)
    x_cam = normalize(np.cross(up, z_cam))
    
    # Vetor Y da câmera, perpendicular ao plano XZ (produto vetorial de z_cam e x_cam)
    y_cam = np.cross(z_cam, x_cam)

    rotation_matrix = np.column_stack([x_cam, y_cam, z_cam])

    # Aplicando a transformação nos vetores normais (multiplicação matricial)
    transformed_normals = np.dot(normal, rotation_matrix.T)  # Transposta de R para a transformação correta

    return transformed_normals


In [7]:
#Vector Conversion
def vector_to_euler(vec):
    vec = np.array(vec) / np.linalg.norm(vec)  # Normalize the vector
    reference = np.array([1, 0, 0])  # Reference vector
    
    if np.allclose(vec, reference):
        return np.array([0.0, 0.0, 0.0])
    
    axis = np.cross(reference, vec)
    angle = np.arccos(np.dot(reference, vec))
    
    if np.linalg.norm(axis) < 1e-6:
        axis = np.array([0, 0, 1])
    else:
        axis = axis / np.linalg.norm(axis)
    
    rotation = R.from_rotvec(angle * axis)
    print(rotation)
    euler_angles = rotation.as_euler('xyz')
    
    return euler_angles

def quaternion_from_vectors(v_from, v_to):
    # Normaliza:
    v_from = v_from / np.linalg.norm(v_from)
    v_to   = v_to   / np.linalg.norm(v_to)
    
    # dot product -> cosseno do ângulo:
    dot_val = np.dot(v_from, v_to)
    # clamp para evitar problema se cair fora de [-1, 1] por ruído
    dot_val = np.clip(dot_val, -1.0, 1.0)
    
    theta = np.arccos(dot_val)  # ângulo escalar
    
    # Se o ângulo for pequeno, retorna identidade
    if abs(theta) < 1e-8:
        return np.array([1, 0, 0, 0], dtype=float)
    
    # Se o ângulo estiver perto de 180º:
    if abs(theta - np.pi) < 1e-8:
        # Precisamos de um eixo ortogonal a v_from
        # Pega um qualquer ortogonal a v_from
        # Exemplo: eixos canônicos e escolhe o que for menos paralelo
        axis_candidates = [np.array([1,0,0], dtype=float),
                           np.array([0,1,0], dtype=float),
                           np.array([0,0,1], dtype=float)]
        best_axis = None
        best_dot = 1.0
        for c in axis_candidates:
            d_ = abs(np.dot(c, v_from))
            if d_ < best_dot:
                best_dot = d_
                best_axis = c
        axis = np.cross(v_from, best_axis)
        axis /= np.linalg.norm(axis)
    else:
        # Eixo = cross(v_from, v_to)
        axis = np.cross(v_from, v_to)
        axis /= np.linalg.norm(axis)
    
    half_angle = theta / 2
    w = np.cos(half_angle)
    sin_h = np.sin(half_angle)
    x, y, z = axis * sin_h
    return np.array([w, x, y, z], dtype=float)

## Load View Data

In [8]:
# Load View Data
data_root = 'data/'

views = {
    "default": {
        "depth": "def_depth.npy",
        "normal_map": "def_normal_map.npy",
        "img": "def_rgb_cylinder.png",
        "coords": "def_cylinder_pixel_coords.npy",
        "cam_pos": [3, -1.5, 0.2],
        "up": [0, 0, 1]
    },
    "top": {
        "depth": "top_depth.npy",
        "normal_map": "top_normal_map.npy",
        "img": "top_rgb_cylinder.png",
        "coords": "top_cylinder_pixel_coords.npy",
        "cam_pos": [0, 0, 1],
        "up": [0, 1, 0] # must be different from Z axis 
    },
    "side": {
        "depth": "side_depth.npy",
        "normal_map": "side_normal_map.npy",
        "img": "side_rgb_cylinder.png",
        "coords": "side_cylinder_pixel_coords.npy",
        "cam_pos": [1, 0, 0.05],
        "up": [0, 0, 1]
    },
    "iso": {
        "depth": "iso_depth.npy",
        "normal_map": "iso_normal_map.npy",
        "img": "iso_rgb_cylinder.png",
        "coords": "iso_cylinder_pixel_coords.npy",
        "cam_pos": [1, 1, 1],
        "up": [0, 0, 1]
    }
}

#------------Cam Static Values-------------#
cam_lookat_0 = [0.0, 0.0, 0.05]
cam_lookat = cam_lookat_0
fov = 30
img_width, img_height = 1280, 960
#------------------------------------------#

#------------Cilinder-------------#
cylinder.set_pos(cam_lookat)
cylinder.set_quat(quaternion_from_euler(0,0,0,'ryxz'))
#---------------------------------#

dv=[0,0,0]


[Genesis] [17:08:39] [WARNING] Base link is fixed. Overriding base link pose.
[Genesis] [17:08:39] [WARNING] Base link is fixed. Overriding base link pose.


In [9]:
# Move Scene
dx = 0.0
dy = 0.0
dz = 0.0
dv=[dx,dy,dz]

print(f"Deslocamento: \033[1;92m{dv}\033[0m")

Deslocamento: [0.0, 0.0, 0.0]


In [10]:
# Side View

# Camera
cam_pos = views['side']['cam_pos']
up = views['side']['up']
cam_lookat = cam_lookat_0

for i in range(3):
    cam_pos[i]=cam_pos[i]+dv[i]
    cam_lookat[i]=cam_lookat[i]+dv[i]

#Load Data
depth_path = data_root+views['side']['depth']
normal_path = data_root+views['side']['img']
cylinder_pixels = data_root+views['side']['coords']

cylinder_pixel_coords = np.load(cylinder_pixels)
depth = np.load(depth_path)

# Compute Normal Map using D2NT
normal_map_d2nt = d2nt(depth)

In [11]:
# Isometric View

# Camera
cam_pos = views['iso']['cam_pos']
up = views['iso']['up']
cam_lookat = cam_lookat_0

for i in range(3):
    cam_pos[i]=cam_pos[i]+dv[i]
    cam_lookat[i]=cam_lookat[i]+dv[i]

#Load Data
depth_path = data_root+views['iso']['depth']
normal_path = data_root+views['iso']['img']
cylinder_pixels = data_root+views['iso']['coords']

cylinder_pixel_coords = np.load(cylinder_pixels)
depth = np.load(depth_path)

# Compute Normal Map using D2NT
normal_map_d2nt = d2nt(depth)

In [12]:
# Top View

# Camera
cam_pos = views['top']['cam_pos']
up = views['top']['up']
cam_lookat = cam_lookat_0

for i in range(3):
    cam_pos[i]=cam_pos[i]+dv[i]
    cam_lookat[i]=cam_lookat[i]+dv[i]

#Load Data
depth_path = data_root+views['top']['depth']
normal_path = data_root+views['top']['img']
cylinder_pixels = data_root+views['top']['coords']

cylinder_pixel_coords = np.load(cylinder_pixels)
depth = np.load(depth_path)

# Compute Normal Map using D2NT
normal_map_d2nt = d2nt(depth)

In [13]:
# Default View

# Camera
cam_pos = views['default']['cam_pos']
up = views['default']['up']
cam_lookat = cam_lookat_0

for i in range(3):
    cam_pos[i]=cam_pos[i]+dv[i]
    cam_lookat[i]=cam_lookat[i]+dv[i]

#Load Data
depth_path = data_root+views['default']['depth']
normal_path = data_root+views['default']['img']
cylinder_pixels = data_root+views['default']['coords']

cylinder_pixel_coords = np.load(cylinder_pixels)
depth = np.load(depth_path)

# Compute Normal Map using D2NT
normal_map_d2nt = d2nt(depth)

In [14]:
# Random View
cam_lookat = cam_lookat_0
up = [0, 0, 1]

#randomize camera position
rand_cam_x = random.uniform(-0.5, 0.5)
rand_cam_y = random.uniform(-0.5, 0.5)
rand_cam_z = random.uniform(0.16, 0.6)
cam_pos=[rand_cam_x,rand_cam_y,rand_cam_z]

#get camera info
cam.set_pose(
    pos = (rand_cam_x,rand_cam_y,rand_cam_z),
    lookat = cam_lookat,
)

for i in range(5):
    scene.step()
    render_output = cam.render(rgb=True, depth=True, segmentation=True, normal=True, colorize_seg=False, )

rgb, depth, seg, normal_map = render_output

for i in range(3):
    cam_pos[i]=cam_pos[i]+dv[i]
    cam_lookat[i]=cam_lookat[i]+dv[i]
    
# Compute Normal Map using D2NT
normal_map_d2nt = d2nt(depth)

# Get array within image that contains the object
i = 0
for w in range(img_width):
    for h in range(img_height):
        i=0

cylinder_pixel_coords = np.where(np.array(seg) == 1)
cylinder_pixel_coords=np.transpose(cylinder_pixel_coords)

## Grasp

In [15]:
# Select a random cylinder pixel
random_idx = random.randint(0, len(cylinder_pixel_coords) - 1)
random_y, random_x = cylinder_pixel_coords[random_idx]
specific_point_depth = depth[random_y, random_x]
specific_point_normal = -normal_map_d2nt[random_y, random_x]*0.4  # D2NT normal (already [-1, 1]) com 30 cm de tamanho
print(f"Random pixel coordinate: \033[1;92m({random_x}, {random_y})\033[0m")

specific_point_normal = rotate_vector(specific_point_normal, cam_pos, cam_lookat, up) #vetor com 30 cm de tamanho
quartenion = quaternion_from_euler(* vector_to_euler(-1*specific_point_normal))

grasp_point = pixel_to_world(random_x, random_y, specific_point_depth, cam_pos, cam_lookat, fov, img_width, img_height, up)

position = grasp_point + specific_point_normal

print(f"\033[1;92m{"_______________________________________________________"}\033[0m")

print(f"Orientation: \033[1;92m{euler_from_quaternion(quartenion)}\033[0m")

print(f"\033[1;92m{"_______________________________________________________"}\033[0m")

print(f"Grasp Point: \033[1;92m{grasp_point}\033[0m")

print(f"Normal specific Point: \033[1;92m{specific_point_normal}\033[0m")

print(f"Gripper Graspping Point: \033[1;92m{position}\033[0m")

print(f"\033[1;92m{"_______________________________________________________"}\033[0m")

cylinder.set_pos(cam_lookat)
camera.set_pos(cam_pos)
camera.set_quat(quaternion_from_vectors(cam_lookat,cam_pos))

for _ in range(1):
    scene.step()

spot_gripper.set_quat(quartenion)
spot_gripper.set_pos(position)

print(f"Gripper Position: {spot_gripper.get_pos()}")

for _ in range(1):
    scene.step()  # Offset along normal

[Genesis] [17:08:41] [WARNING] Base link is fixed. Overriding base link pose.
[Genesis] [17:08:41] [WARNING] Base link is fixed. Overriding base link pose.
[Genesis] [17:08:41] [WARNING] Base link is fixed. Overriding base link pose.
[Genesis] [17:08:41] [WARNING] Base link is fixed. Overriding base link pose.
[Genesis] [17:08:41] [WARNING] Base link is fixed. Overriding base link pose.


Random pixel coordinate: (715, 316)
_______________________________________________________
Orientation: (0.41577767243097424, 1.5705229349776328, 0.4158881095983314)
_______________________________________________________
Grasp Point: [-0.02723348 -0.00072369  0.1006765 ]
Normal specific Point: [-1.00034926e-04 -4.41803898e-05  3.99999984e-01]
Gripper Graspping Point: [-0.02733352 -0.00076787  0.50067648]
_______________________________________________________
Gripper Position: tensor([-0.0273, -0.0008,  0.5007], device='cuda:0')
